In [1]:
import pandas as pd
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(
    r"D:\PROJECTS\Customer_Review_Intelligence\.env"
)

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

df = pd.read_csv(
    "../data/processed/harmony_reviews_with_clusters.csv"
)

print(df.shape)

(1604, 34)


In [2]:
#Aspect sentiment function
def aspect_sentiment(review):

    prompt = f"""
Analyze this customer review of the Logitech Harmony Ultimate One.

Identify the product aspects explicitly mentioned.

For each aspect, provide:

Aspect | Sentiment | Evidence

Sentiment must be one of:
Positive
Negative
Neutral

Rules:
- Only identify aspects explicitly supported by the review.
- Do not invent information.
- Keep evidence short.
- A review may contain multiple aspects.

Review:
{review}
"""

    response = client.responses.create(
        model="gpt-6-luna",
        input=prompt
    )

    return response.output_text.strip()

In [3]:
#Test
review = df["original_text"].iloc[0]

print(review)

print("\n--- ASPECT SENTIMENT ---\n")

print(
    aspect_sentiment(review)
)

Cannot hit play, FF or Rev, etc, without having your hand/arm over the touch pad which is VERY sensitive! The slightest brush can mess up what you are watching. I have to brace my forearm on the bottom part, keep my wrist bent up very high in order to use the playing options. Even then it's not a given it won't sense your arm over the top and change it to a different channel or activity. Also no way to use a number pad that I can find so have to grab another remote to do that. It's a nice idea but very flawed design. It would be much better to have the touch screen at the top at least, and a screen to go to that gives the number/alphabet option.

--- ASPECT SENTIMENT ---

| Aspect | Sentiment | Evidence |
|---|---|---|
| Touchpad sensitivity | Negative | “VERY sensitive”; a slight brush can disrupt viewing. |
| Using playback controls | Negative | Reaching play, fast-forward, or rewind requires awkward arm and wrist positioning. |
| Accidental channel or activity changes | Negative | T

In [4]:
#aspect sentiment on 10 reviews
aspect_df = df.sample(
    10,
    random_state=42
).copy()

aspect_df["aspect_sentiment"] = aspect_df[
    "original_text"
].apply(aspect_sentiment)

print("Aspect sentiment completed.")

Aspect sentiment completed.


In [5]:
aspect_df.to_csv(
    "../outputs/aspect_sentiment_results.csv",
    index=False
)

print("Saved aspect sentiment results.")

Saved aspect sentiment results.
